# FedVocabHAR PAMAP2 Phase 4C

Frozen ranked-MNN exploratory evaluation.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import precision_recall_fscore_support

PROJECT=Path('/content/drive/MyDrive/FedVocabHAR');P4A=PROJECT/'artifacts'/'phase4a_fixed3';P4B=PROJECT/'artifacts'/'phase4b';P3D=PROJECT/'artifacts'/'phase3d';OUT=PROJECT/'artifacts'/'phase4c';OUT.mkdir(parents=True,exist_ok=True)
manifest=json.loads((P4A/'phase4a_preregistered_manifest.json').read_text());lock=json.loads((P3D/'phase3d_policy_lock.json').read_text());assert lock['selected_policy']=='ranked_mnn'
folds=manifest['outer_folds'];known=tuple(manifest['known_activities']);novel=tuple(manifest['novel_activities']);vocabulary=known+novel
d=np.load(PROJECT/'data'/'pamap2'/'cache'/'pamap2_hand_acc_common8_w128_s64_50hz.npz');x=d['x'].astype(np.float32);subjects=d['subject'].astype(int);activities=d['activity'].astype(str);DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class Block(nn.Module):
    def __init__(self,channels,width=32):
        super().__init__();self.a=nn.Conv1d(channels,width,1,bias=False);self.b=nn.ModuleList([nn.Conv1d(width,width,k,padding=k//2,bias=False) for k in (9,19,39)]);self.p=nn.Sequential(nn.MaxPool1d(3,1,1),nn.Conv1d(channels,width,1,bias=False));self.n=nn.BatchNorm1d(4*width)
    def forward(self,q):
        z=self.a(q);return F.relu(self.n(torch.cat([f(z) for f in self.b]+[self.p(q)],1)))
class Encoder(nn.Module):
    def __init__(self,domains):
        super().__init__();self.b1=Block(4);self.b2=Block(128);self.s=nn.Conv1d(4,128,1,bias=False);self.n=nn.BatchNorm1d(128);self.p=nn.Sequential(nn.Linear(128,128),nn.ReLU(),nn.Linear(128,64));self.decoder=nn.Conv1d(128,4,1);self.domain=nn.Sequential(nn.Linear(64,64),nn.ReLU(),nn.Linear(64,domains))
    def forward(self,q):
        z=F.relu(self.n(self.b2(self.b1(q))+self.s(q)));return F.normalize(self.p(z.mean(-1)),dim=1)
def load(path):
    try:return torch.load(path,map_location='cpu',weights_only=False)
    except TypeError:return torch.load(path,map_location='cpu')
def unit(z):return z/(np.linalg.norm(z,axis=1,keepdims=True)+1e-12)
def encode(model,data):
    out=[];model.eval()
    with torch.no_grad():
        for i in range(0,len(data),512):out.append(model(torch.from_numpy(data[i:i+512]).to(DEVICE)).cpu().numpy())
    return unit(np.concatenate(out))
def cluster(z):
    try:return AgglomerativeClustering(n_clusters=None,distance_threshold=.25,metric='cosine',linkage='average').fit_predict(z)
    except TypeError:return AgglomerativeClustering(n_clusters=None,distance_threshold=.25,affinity='cosine',linkage='average').fit_predict(z)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
def adapt_checkpoint_keys(state):
    adapted = {}

    for key, value in state.items():
        new_key = key

        new_key = new_key.replace("block1.bottleneck.", "b1.a.")
        new_key = new_key.replace("block1.branches.", "b1.b.")
        new_key = new_key.replace("block1.pool.", "b1.p.")
        new_key = new_key.replace("block1.bn.", "b1.n.")

        new_key = new_key.replace("block2.bottleneck.", "b2.a.")
        new_key = new_key.replace("block2.branches.", "b2.b.")
        new_key = new_key.replace("block2.pool.", "b2.p.")
        new_key = new_key.replace("block2.bn.", "b2.n.")

        new_key = new_key.replace("skip.", "s.")
        new_key = new_key.replace("project.", "p.")
        new_key = new_key.replace("bn.", "n.")

        adapted[new_key] = value

    return adapted

results=[];promoted=[]
for fold in folds:
    fold_id=int(fold['fold']);bundle=load(P4B/'folds'/f'fold_{fold_id}_models.pt');stats=bundle['stats']
    inp=np.concatenate([(x[:,:3]-stats['xyz_mean'])/stats['xyz_std'],(x[:,3:4]-stats['mag_mean'])/stats['mag_std']],1).astype(np.float32)
    model = Encoder(
        len(fold["train_subjects"])
    ).to(DEVICE)

    state = adapt_checkpoint_keys(
        bundle["anchor_state_dict"]
    )

    model.load_state_dict(
        state,
        strict=True
    )

    embedding = encode(
        model,
        inp
    )
    train=tuple(fold['train_subjects']);integration=tuple(fold['integration_subjects']);test=tuple(fold['test_subjects'])
    known_idx=np.where(np.isin(subjects,train)&np.isin(activities,known))[0];known_y=np.asarray([known.index(a) for a in activities[known_idx]])
    known_prototypes=np.asarray([unit(embedding[known_idx][known_y==i].mean(0,keepdims=True))[0] for i in range(4)])
    signatures=[]
    for client in integration:
        stream=np.where(subjects==client)[0];ranked=stream[np.argsort(1-(embedding[stream]@known_prototypes.T).max(1))[-min(600,len(stream)):]];labels=cluster(embedding[ranked])
        for label in np.unique(labels):
            windows=ranked[labels==label]
            if len(windows)<12:continue
            signature=unit(embedding[windows].mean(0,keepdims=True))[0];medoid=int(windows[np.argmax(embedding[windows]@signature)])
            signatures.append({'client':int(client),'signature':signature,'windows':windows,'medoid':medoid})
    pairs=[]
    for i,left in enumerate(signatures):
        candidates=[j for j,right in enumerate(signatures) if right['client']!=left['client'] and left['signature']@right['signature']>=.89]
        if not candidates:continue
        j=max(candidates,key=lambda q:left['signature']@signatures[q]['signature']);back=[k for k,item in enumerate(signatures) if item['client']!=signatures[j]['client'] and signatures[j]['signature']@item['signature']>=.89]
        if back and max(back,key=lambda q:signatures[j]['signature']@signatures[q]['signature'])==i:pairs.append(tuple(sorted((i,j))))
    candidates=[]
    for i,j in sorted(set(pairs)):
        left,right=signatures[i],signatures[j];signature=unit((left['signature']+right['signature'])[None,:])[0];medoid=left['medoid'] if signature@left['signature']>=signature@right['signature'] else right['medoid'];label=activities[medoid]
        if label in novel:candidates.append({'signature':signature,'label':label,'clients':{left['client'],right['client']},'support':len(left['windows'])+len(right['windows'])})
    queried=sorted(candidates,key=lambda q:q['support'],reverse=True)[:4];entries=[];seen=set()
    for entry in queried:
        if entry['label'] not in seen:entries.append(entry);seen.add(entry['label'])
    test_idx=np.where(np.isin(subjects,test))[0];labels=list(known)+[e['label'] for e in entries];prototypes=np.asarray([*known_prototypes]+[e['signature'] for e in entries]);prediction=np.asarray(labels)[(embedding[test_idx]@prototypes.T).argmax(1)]
    f1=precision_recall_fscore_support(activities[test_idx],prediction,labels=list(vocabulary),zero_division=0)[2]
    results.append({'fold':fold_id,'label_queries':len(queried),'promoted_entries':len(entries),'novel_recovery':len(entries)/4,'novel_macro_f1':float(f1[4:].mean()),'combined_macro_f1':float(f1.mean())})
    for n,entry in enumerate(entries,1):promoted.append({'fold':fold_id,'entry':n,'label':entry['label'],'clients':','.join(map(str,sorted(entry['clients']))),'support':entry['support']})
results=pd.DataFrame(results);entries=pd.DataFrame(promoted);results.to_csv(OUT/'phase4c_ranked_mnn_results.csv',index=False);entries.to_csv(OUT/'phase4c_promoted_entries.csv',index=False)
decision={'phase':'Phase 4C','status':'Exploratory frozen-model analysis. No model training, threshold change, or policy selection occurred.','policy':'ranked_mnn','mean_results':results.mean(numeric_only=True).to_dict()};(OUT/'phase4c_exploratory_decision.json').write_text(json.dumps(decision,indent=2));display(results);print(json.dumps(decision,indent=2))


,fold,label_queries,promoted_entries,novel_recovery,novel_macro_f1,combined_macro_f1
0,1,0,0,0.00,0.000000,0.245577
1,2,1,1,0.25,0.092566,0.371851
2,3,1,1,0.25,0.130009,0.420366
3,4,1,1,0.25,0.061710,0.256753


{
  "phase": "Phase 4C",
  "status": "Exploratory frozen-model analysis. No model training, threshold change, or policy selection occurred.",
  "policy": "ranked_mnn",
  "mean_results": {
    "fold": 2.5,
    "label_queries": 0.75,
    "promoted_entries": 0.75,
    "novel_recovery": 0.1875,
    "novel_macro_f1": 0.07107132894430573,
    "combined_macro_f1": 0.3236366559286852
  }
}
